In [5]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score
)
from sklearn.neighbors import NearestNeighbors
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from transformers import TFBertModel, BertTokenizer
from sklearn.model_selection import train_test_split
import os


data_dir = r"."  # 数据和模型所在目录
csv_files = ["goemotions_1.csv", "goemotions_2.csv", "goemotions_3.csv"]
model_path = os.path.join(data_dir, "goemotions_bert_model")  # GoEmotions BERT 模型文件夹
output_dir = data_dir
os.makedirs(output_dir, exist_ok=True)

# 固定随机种子
tf.random.set_seed(42)
np.random.seed(42)

# === 1. 加载并预处理数据===
print("Loading and preprocessing data...")
dataframes = [pd.read_csv(os.path.join(data_dir, file)) for file in csv_files]
data = pd.concat(dataframes, ignore_index=True)

emotion_columns = [
    'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring',
    'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval',
    'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief',
    'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization',
    'relief', 'remorse', 'sadness', 'surprise', 'neutral'
]
depression_emotions = ['sadness', 'grief', 'remorse', 'disappointment', 'disapproval']
positive_emotions = ['admiration', 'amusement', 'approval', 'caring', 'excitement',
                     'gratitude', 'joy', 'love', 'optimism', 'pride', 'relief']
neutral_emotions = [
    'anger', 'annoyance', 'confusion', 'curiosity', 'desire',
    'disgust', 'embarrassment', 'fear', 'nervousness', 'realization', 'surprise', 'neutral'
]

# 按 id 聚合
agg_dict = {col: 'sum' for col in emotion_columns}
agg_dict.update({
    'text': 'first', 'author': 'first', 'subreddit': 'first',
    'example_very_unclear': 'first'
})
data = data.groupby('id').agg(agg_dict).reset_index()

# 转为 binary
for col in emotion_columns:
    data[col] = (data[col] >= 1).astype(int)

# 过滤 unclear
data = data[data['example_very_unclear'] == False]

# 稀有情绪同意度过滤
rare_emotions = ['grief', 'pride', 'relief', 'nervousness']
data['rare_sum'] = data[rare_emotions].sum(axis=1)
data = data[(data[emotion_columns].sum(axis=1) >= 1) | (data['rare_sum'] >= 2)]
data = data.drop(columns=['rare_sum'])

# 按 author 聚合
author_data = data.groupby('author').agg({
    **{col: 'sum' for col in emotion_columns},
    'text': lambda x: ' '.join(x),
    'subreddit': 'first',
    'id': 'count'
}).rename(columns={'id': 'comment_count'}).reset_index()

author_data = author_data[author_data['comment_count'] >= 2]
print(f"Filtered to authors with >=2 comments: {len(author_data)}")

# 计算 DEDI 并生成标签（prevalence=11.02%）
author_data['DEDI'] = (
    author_data[depression_emotions].sum(axis=1) /
    (author_data[depression_emotions].sum(axis=1) +
     author_data[positive_emotions].sum(axis=1) +
     author_data[neutral_emotions].sum(axis=1) + 1e-8)
)

prevalence = 0.1102
threshold = author_data['DEDI'].quantile(1 - prevalence)
author_data['label'] = (author_data['DEDI'] >= threshold).astype(int)


train_authors, test_authors = train_test_split(
    author_data, test_size=0.2, random_state=42, stratify=author_data['label']
)

print(f"Train authors: {len(train_authors)}, Test authors: {len(test_authors)}")

# 构建 length-matched 测试集
test_authors_copy = test_authors.copy()
test_authors_copy['total_chars'] = test_authors_copy['text'].str.len()

DEDI_df = test_authors_copy[test_authors_copy['label'] == 1]
non_df  = test_authors_copy[test_authors_copy['label'] == 0]

nn = NearestNeighbors(n_neighbors=1)
nn.fit(DEDI_df[['total_chars']].values)
distances, indices = nn.kneighbors(non_df[['total_chars']].values)

threshold_dist = 8000
mask = distances.flatten() <= threshold_dist
matched_non = non_df.iloc[mask].copy()
matched_DEDI = DEDI_df.iloc[indices.flatten()[mask]].copy()

min_size = min(len(matched_DEDI), len(matched_non))
matched_DEDI = matched_DEDI.sample(n=min_size, random_state=42)
matched_non  = matched_non.sample(n=min_size, random_state=42)

test_matched = pd.concat([matched_DEDI, matched_non]).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Length-matched test set: {len(test_matched)} samples ({test_matched['label'].sum()} depressed)")

y_true = test_matched['label'].values

# === 3. 在训练集上 fit 所有特征提取器 ===
print("Fitting feature extractors on training authors...")

# TF-IDF
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=1000)
tfidf.fit(train_authors['text'])

# CountVectorizer
count_vec = CountVectorizer(ngram_range=(1, 2), max_features=1000)
count_vec.fit(train_authors['text'])

# Frozen BERT embeddings
tokenizer = BertTokenizer.from_pretrained(model_path)
bert_model_base = TFBertModel.from_pretrained(model_path)
bert_model_base.trainable = False

def get_bert_pooler_embeddings(texts):
    encodings = tokenizer(texts.tolist(), max_length=100, padding='max_length',
                         truncation=True, return_tensors='tf')
    dataset = tf.data.Dataset.from_tensor_slices({
        'input_ids': encodings['input_ids'],
        'attention_mask': encodings['attention_mask'],
        'token_type_ids': encodings.get('token_type_ids', tf.zeros_like(encodings['input_ids']))
    }).batch(16)
    embeddings = []
    for batch in dataset:
        outputs = bert_model_base(batch)
        embeddings.append(outputs.pooler_output.numpy())
    return np.vstack(embeddings)

train_bert_emb = get_bert_pooler_embeddings(train_authors['text'])

# Subreddit dummies
train_dummies = pd.get_dummies(train_authors['subreddit'], dtype=int)

# === 4. 训练并评估所有基线模型 ===
results = []

X_test_tfidf = tfidf.transform(test_matched['text']).toarray()
X_test_count = count_vec.transform(test_matched['text']).toarray()
test_bert_emb = get_bert_pooler_embeddings(test_matched['text'])
test_dummies = pd.get_dummies(test_matched['subreddit'], dtype=int)
test_dummies = test_dummies.reindex(columns=train_dummies.columns, fill_value=0)

# 1. TF-IDF + LR
print("Running TF-IDF + LR...")
X_train_tfidf = tfidf.transform(train_authors['text']).toarray()
clf = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
clf.fit(X_train_tfidf, train_authors['label'])
prob = clf.predict_proba(X_test_tfidf)[:, 1]
pred = (prob >= 0.5).astype(int)
results.append({'Method': 'TF-IDF + LR',
                'AUC': round(roc_auc_score(y_true, prob), 4),
                'F1': round(f1_score(y_true, pred), 4),
                'Precision': round(precision_score(y_true, pred), 4),
                'Recall': round(recall_score(y_true, pred), 4),
                'Accuracy': round(accuracy_score(y_true, pred), 4)})

# 2. TF-IDF + SVM
print("Running TF-IDF + SVM...")
clf = SVC(probability=True, class_weight='balanced', random_state=42)
clf.fit(X_train_tfidf, train_authors['label'])
prob = clf.predict_proba(X_test_tfidf)[:, 1]
pred = (prob >= 0.5).astype(int)
results.append({'Method': 'TF-IDF + SVM',
                'AUC': round(roc_auc_score(y_true, prob), 4),
                'F1': round(f1_score(y_true, pred), 4),
                'Precision': round(precision_score(y_true, pred), 4),
                'Recall': round(recall_score(y_true, pred), 4),
                'Accuracy': round(accuracy_score(y_true, pred), 4)})

# 3. TF-IDF + RF
print("Running TF-IDF + RF...")
clf = RandomForestClassifier(class_weight='balanced', random_state=42)
clf.fit(X_train_tfidf, train_authors['label'])
prob = clf.predict_proba(X_test_tfidf)[:, 1]
pred = (prob >= 0.5).astype(int)
results.append({'Method': 'TF-IDF + RF',
                'AUC': round(roc_auc_score(y_true, prob), 4),
                'F1': round(f1_score(y_true, pred), 4),
                'Precision': round(precision_score(y_true, pred), 4),
                'Recall': round(recall_score(y_true, pred), 4),
                'Accuracy': round(accuracy_score(y_true, pred), 4)})

# 4. CountVec + NB
print("Running CountVec + NB...")
X_train_count = count_vec.transform(train_authors['text']).toarray()
clf = MultinomialNB()
clf.fit(X_train_count, train_authors['label'])
prob = clf.predict_proba(X_test_count)[:, 1]
pred = (prob >= 0.5).astype(int)
results.append({'Method': 'CountVec + NB',
                'AUC': round(roc_auc_score(y_true, prob), 4),
                'F1': round(f1_score(y_true, pred), 4),
                'Precision': round(precision_score(y_true, pred), 4),
                'Recall': round(recall_score(y_true, pred), 4),
                'Accuracy': round(accuracy_score(y_true, pred), 4)})

# 5. CountVec + LR
print("Running CountVec + LR...")
clf = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
clf.fit(X_train_count, train_authors['label'])
prob = clf.predict_proba(X_test_count)[:, 1]
pred = (prob >= 0.5).astype(int)
results.append({'Method': 'CountVec + LR',
                'AUC': round(roc_auc_score(y_true, prob), 4),
                'F1': round(f1_score(y_true, pred), 4),
                'Precision': round(precision_score(y_true, pred), 4),
                'Recall': round(recall_score(y_true, pred), 4),
                'Accuracy': round(accuracy_score(y_true, pred), 4)})

# 6. Vanilla BERT (frozen + LR)
print("Running Vanilla BERT...")
clf = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
clf.fit(train_bert_emb, train_authors['label'])
prob = clf.predict_proba(test_bert_emb)[:, 1]
pred = (prob >= 0.5).astype(int)
results.append({'Method': 'Vanilla BERT',
                'AUC': round(roc_auc_score(y_true, prob), 4),
                'F1': round(f1_score(y_true, pred), 4),
                'Precision': round(precision_score(y_true, pred), 4),
                'Recall': round(recall_score(y_true, pred), 4),
                'Accuracy': round(accuracy_score(y_true, pred), 4)})

# 7. Subreddit-only
print("Running Subreddit-only...")
clf = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
clf.fit(train_dummies, train_authors['label'])
prob = clf.predict_proba(test_dummies)[:, 1]
pred = (prob >= 0.5).astype(int)
results.append({'Method': 'Subreddit-only',
                'AUC': round(roc_auc_score(y_true, prob), 4),
                'F1': round(f1_score(y_true, pred), 4),
                'Precision': round(precision_score(y_true, pred), 4),
                'Recall': round(recall_score(y_true, pred), 4),
                'Accuracy': round(accuracy_score(y_true, pred), 4)})

# === 5. 输出完整结果 ===
baseline_df = pd.DataFrame(results)
print("\n" + "="*90)
print("Complete Baseline12 Results (Independent Length-Matched Test Set)")
print("="*90)
print(baseline_df.to_string(index=False))

# 保存结果
baseline_df.to_csv(os.path.join(output_dir, 'baseline12_full_reproduced_results.csv'), index=False)
print(f"\nResults saved to: baseline12_full_reproduced_results.csv")

Loading and preprocessing data...
Filtered to authors with >=2 comments: 5182
Train authors: 4145, Test authors: 1037
Length-matched test set: 1820 samples (910 depressed)
Fitting feature extractors on training authors...


Some layers from the model checkpoint at .\goemotions_bert_model were not used when initializing TFBertModel: ['classifier', 'dropout_227']
- This IS expected if you are initializing TFBertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
All the layers of TFBertModel were initialized from the model checkpoint at .\goemotions_bert_model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions without further training.


Running TF-IDF + LR...
Running TF-IDF + SVM...
Running TF-IDF + RF...
Running CountVec + NB...
Running CountVec + LR...
Running Vanilla BERT...
Running Subreddit-only...

Complete Baseline12 Results (Independent Length-Matched Test Set)
        Method    AUC     F1  Precision  Recall  Accuracy
   TF-IDF + LR 0.6570 0.5010     0.6285  0.4165    0.5852
  TF-IDF + SVM 0.6588 0.0409     1.0000  0.0209    0.5104
   TF-IDF + RF 0.6291 0.0800     0.9500  0.0418    0.5198
 CountVec + NB 0.6572 0.2363     0.7697  0.1396    0.5489
 CountVec + LR 0.6155 0.4896     0.6282  0.4011    0.5819
  Vanilla BERT 0.7346 0.5965     0.7297  0.5044    0.6588
Subreddit-only 0.4939 0.4124     0.4837  0.3593    0.4879

Results saved to: baseline12_full_reproduced_results.csv
